# Testing out the ENTSOE API

In [1]:
# import packages

import os
import pandas as pd
from dotenv import load_dotenv
from entsoe import EntsoePandasClient

In [2]:
# set up ENTSOE API key and client

load_dotenv()
api_key = os.getenv("ENTSOE_API_KEY")
assert api_key, "ENTSOE_API_KEY not found"
client = EntsoePandasClient(api_key=api_key)

In [3]:
# settings for basic DE_LU test call

zone = "DE_LU"
start = pd.Timestamp("2024-06-03", tz="Europe/Berlin")
end = pd.Timestamp("2024-06-10", tz="Europe/Berlin")

In [6]:
# helper to pull descriptive data from download

def describe(obj, name):
    print(f"{name}: {type(obj).__name__}, shape {obj.shape}")
    print("tz:", obj.index.tz)
    print("time steps:", obj.index.to_series().diff().value_counts().to_dict())
    print("duplicate timestamps:", obj.index.duplicated().sum())
    if isinstance(obj, pd.DataFrame):
        print("columns:", list(obj.columns))
    display(obj.head())

In [7]:
# day ahead price test

prices = client.query_day_ahead_prices(zone, start=start, end=end)
describe(prices, "day-ahead price")

day-ahead price: Series, shape (169,)
tz: Europe/Berlin
time steps: {Timedelta('0 days 01:00:00'): 168}
duplicate timestamps: 0


2024-06-03 00:00:00+02:00    98.26
2024-06-03 01:00:00+02:00    84.58
2024-06-03 02:00:00+02:00    83.67
2024-06-03 03:00:00+02:00    83.81
2024-06-03 04:00:00+02:00    82.17
Freq: h, dtype: float64

In [9]:
# load forecast test

load_forecast = client.query_load_forecast(zone, start=start, end=end)
describe(load_forecast, "load forecast")

load forecast: DataFrame, shape (672, 1)
tz: Europe/Berlin
time steps: {Timedelta('0 days 00:15:00'): 671}
duplicate timestamps: 0
columns: ['Forecasted Load']


,Forecasted Load
2024-06-03 00:00:00+02:00,41417.04
2024-06-03 00:15:00+02:00,41008.25
2024-06-03 00:30:00+02:00,40676.67
2024-06-03 00:45:00+02:00,40479.41
2024-06-03 01:00:00+02:00,40045.24


In [10]:
# wind and solare forecast test

wind_solar_forecast = client.query_wind_and_solar_forecast(zone, start=start, end=end)
describe(wind_solar_forecast, "wind and solar forecast")

wind and solar forecast: DataFrame, shape (672, 3)
tz: Europe/Berlin
time steps: {Timedelta('0 days 00:15:00'): 671}
duplicate timestamps: 0
columns: ['Solar', 'Wind Offshore', 'Wind Onshore']


,Solar,Wind Offshore,Wind Onshore
2024-06-03 00:00:00+02:00,0.0,3318.80,12214.46
2024-06-03 00:15:00+02:00,0.0,3247.00,11977.49
2024-06-03 00:30:00+02:00,0.0,3174.31,11732.33
2024-06-03 00:45:00+02:00,0.0,3103.22,11492.70
2024-06-03 01:00:00+02:00,0.0,3047.46,11249.56


In [11]:
# load test

load = client.query_load(zone, start=start, end=end)
describe(load, "load")

load: DataFrame, shape (672, 1)
tz: Europe/Berlin
time steps: {Timedelta('0 days 00:15:00'): 671}
duplicate timestamps: 0
columns: ['Actual Load']


,Actual Load
2024-06-03 00:00:00+02:00,41756.02
2024-06-03 00:15:00+02:00,41481.72
2024-06-03 00:30:00+02:00,40860.68
2024-06-03 00:45:00+02:00,40402.65
2024-06-03 01:00:00+02:00,40066.20


In [12]:
# installed generation capacity test

# capacity only available once a year so default week of start and end date won't work
capacity = client.query_installed_generation_capacity(zone, 
                                                      start=pd.Timestamp("2024-01-01", tz="Europe/Berlin"), 
                                                      end=pd.Timestamp("2025-01-01", tz="Europe/Berlin"))
describe(capacity, "capacity")

capacity: DataFrame, shape (1, 19)
tz: Europe/Berlin
time steps: {}
duplicate timestamps: 0
columns: ['Biomass', 'Fossil Brown coal/Lignite', 'Fossil Coal-derived gas', 'Fossil Gas', 'Fossil Hard coal', 'Fossil Oil', 'Fossil Oil shale', 'Fossil Peat', 'Geothermal', 'Hydro Pumped Storage', 'Hydro Run-of-river and pondage', 'Hydro Water Reservoir', 'Marine', 'Other', 'Other renewable', 'Solar', 'Waste', 'Wind Offshore', 'Wind Onshore']


,Biomass,Fossil Brown coal/Lignite,Fossil Coal-derived gas,Fossil Gas,Fossil Hard coal,Fossil Oil,Fossil Oil shale,Fossil Peat,Geothermal,Hydro Pumped Storage,Hydro Run-of-river and pondage,Hydro Water Reservoir,Marine,Other,Other renewable,Solar,Waste,Wind Offshore,Wind Onshore
2024-01-02 00:00:00+01:00,8659.54,18365.0,1257.0,36327.0,18411.0,4118.0,0.0,0.0,54.0,9449.0,3752.8,1392.0,0.0,1630.0,356.45,77015.56,1832.0,8456.0,60048.62


## Main findings

- Price data initially in one hour chunks but switches to 15-minute periods in October 2025
- Most other data is in 15-minute periods so will have to be collapsed into hourly values to make it usable against price
- Capacity only available once a year
- Data formats are quite different other than datetime column so further cleaning and consolidation will be needed